# Lab 9A: Autoencoders — What Does a Network Learn Without Labels?

**Before you start: Runtime → Change runtime type → GPU.**

This week's lab consists of **four notebooks** that follow the story of the lecture: each model is
motivated by a weakness of the previous one.

| Notebook | Question | Time |
|---|---|---|
| **9A** Shallow and sparse autoencoders, and a 2D embedding | What does a network learn without labels? | 45 min |
| **9B** Convolutional autoencoders, denoising and skip connections | What does the latent space look like, and what is it good for? | 45 min |
| **9C** Variational autoencoders | Can we *generate* new images by sampling the latent space? | 50 min |
| **9D** GANs and conditional GANs | Why are VAE samples blurry, and what does it cost to fix that? | 40 min |

**Theory ↔ code.** Throughout the notebooks, boxes marked **🔗 Theory ↔ code** show how the
equations from the lectures map to lines of PyTorch. Use them: at the exam you should be able to go
both ways, from an equation to the code that implements it, and from a line of code to the concept
behind it.

**Cells marked 🛠️ TASK contain gaps (`???`) that you must fill in.** Most come with a test cell; don't
move on until the tests pass. As in Labs 6–8, write down your ✍️ **Prediction** before running an
experiment, and focus on **explaining** what you observe.

### ⚠️ The limits of this lab

To keep training times to a minute or two, we use **10% of MNIST** (6,000 training images) and small
networks. MNIST is an easy dataset: almost black-and-white, centred, with little variation. The
*patterns* you will see (gaps in the latent space, blurry VAE samples, unstable GAN training) are
general, but the exact numbers are not.

### This notebook (9A)

| Part | You will… |
|---|---|
| 1. A shallow autoencoder | build a one-hidden-layer autoencoder and its training loop, make it overfit, and see how overfitting (and its remedies) show up in the loss curves and in the learned weights |
| 2. A sparse autoencoder | add a sparsity penalty to the *same* network, and see how it changes the learned features |
| 3. A 2D embedding with a CNN | squeeze a CNN classifier through a 2-dimensional bottleneck, and plot MNIST in 2D |
| 4. Exam-style questions | check your understanding |

## Setup

In [ ]:
import copy
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---- Data: 10% of the MNIST training set, and 2,000 test images (used as a validation set) ----
FRACTION = 0.1
mnist_train = torchvision.datasets.MNIST('./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST('./data', train=False, download=True)
idx = torch.randperm(len(mnist_train), generator=torch.Generator().manual_seed(0))[:int(FRACTION * len(mnist_train))]

# The whole (small) dataset fits on the GPU, so we skip the DataLoader and index tensors directly.
X_train = (mnist_train.data[idx].float() / 255).unsqueeze(1).to(device)    # (6000, 1, 28, 28), values in [0, 1]
y_train = mnist_train.targets[idx].to(device)                               # labels: used ONLY for colouring plots
X_test = (mnist_test.data[:2000].float() / 255).unsqueeze(1).to(device)    # (2000, 1, 28, 28)
y_test = mnist_test.targets[:2000].to(device)
print('X_train:', tuple(X_train.shape), ' X_test:', tuple(X_test.shape))


def batches(n, batch_size=64, shuffle=True):
    # Mini-batches of indices: the same job a DataLoader(shuffle=True) does for you.
    order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    return order.split(batch_size)


def recon_loss(x_hat, x):
    # J = (1/n) * sum_i ||x_hat_i - x_i||^2 : squared error summed over pixels, averaged over the batch
    return ((x_hat - x) ** 2).flatten(1).sum(1).mean()


def show_images(x, nrow=10, title=None, ax=None, figsize=None):
    # Show a batch of (N, 1, 28, 28) images as one grid.
    x = x.detach().float().cpu().clamp(0, 1)
    grid = torchvision.utils.make_grid(x, nrow=nrow, padding=1, pad_value=1)[0]
    if ax is None:
        plt.figure(figsize=figsize or (nrow * 0.6, (len(x) + nrow - 1) // nrow * 0.6))
        ax = plt.gca()
    ax.imshow(grid, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if title:
        ax.set_title(title)

---
# Part 1 — A shallow autoencoder

An autoencoder is trained to copy its input to its output through a **bottleneck**. The simplest
version (the "traditional" autoencoder from the lecture) treats an image as a **vector** and has a
single hidden layer:

$$\mathbf{h} = \sigma(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1), \qquad \hat{\mathbf{x}} = \sigma(\mathbf{W}_2\mathbf{h} + \mathbf{b}_2), \qquad
J = \frac{1}{n}\sum_i \left\|\hat{\mathbf{x}}^{(i)} - \mathbf{x}^{(i)}\right\|^2$$

where $\sigma$ is the sigmoid. Note that the loss compares the output with the **input**: no labels
are needed. This is unsupervised learning.

### 🔗 Theory ↔ code

| Theory | PyTorch |
|---|---|
| image as a vector, $\mathbf{x} \in \mathbb{R}^{784}$ | `x.flatten(1)` (or `nn.Flatten()`): (N, 1, 28, 28) → (N, 784) |
| $\mathbf{W}_1\mathbf{x} + \mathbf{b}_1$ with $H$ hidden units | `nn.Linear(784, H)`. Its `.weight` is $\mathbf{W}_1$ with shape **(H, 784)**: one **row** per hidden unit |
| $\sigma(\cdot)$ | `nn.Sigmoid()` |
| $\hat{\mathbf{x}} = \sigma(\mathbf{W}_2\mathbf{h} + \mathbf{b}_2)$ | `nn.Linear(H, 784)` + `nn.Sigmoid()`, then reshape back to (N, 1, 28, 28) |
| $J$ | `recon_loss(x_hat, x)` (defined in the setup cell) |

### 🛠️ TASK 1a: Build the shallow autoencoder

Fill in the encoder and the decoder. `forward` returns both the reconstruction and the hidden
activations `h` (we need `h` later).

✍️ **Prediction:** how many learnable parameters does the model have for `n_hidden = 196`? Work it out
by hand before you run the test cell.

In [ ]:
class ShallowAE(nn.Module):
    def __init__(self, n_hidden=196):
        super().__init__()
        self.encoder = nn.Sequential(???)      # 784 -> n_hidden, followed by a sigmoid
        self.decoder = nn.Sequential(???)      # n_hidden -> 784, followed by a sigmoid

    def forward(self, x):
        h = self.encoder(x.flatten(1))                    # (N, 784) -> (N, n_hidden)
        x_hat = self.decoder(h).view(-1, 1, 28, 28)       # back to image shape
        return x_hat, h

In [ ]:
# Test for Task 1a
_m = ShallowAE(196).to(device)
_x_hat, _h = _m(X_train[:8])
assert _x_hat.shape == (8, 1, 28, 28), f'reconstruction has shape {tuple(_x_hat.shape)}'
assert _h.shape == (8, 196), f'hidden activations have shape {tuple(_h.shape)}'
assert 0 <= _x_hat.min() and _x_hat.max() <= 1 and 0 <= _h.min() and _h.max() <= 1, 'did you add both sigmoids?'
print('Number of parameters:', sum(p.numel() for p in _m.parameters()))
print('✅ Task 1a passed')

### 🛠️ TASK 1b: Write the training loop

Every model in this lab is trained with the same five lines. Fill them in. Each line corresponds to
a concept from the lectures:

| Line | Concept |
|---|---|
| `opt.zero_grad()` | gradients **accumulate** in PyTorch, so they must be reset before every step (the missing `zero_grad` was one of the bugs in Lab 6) |
| `x_hat, h = model(xb)` | forward pass through the computational graph (Lecture 3) |
| `loss = ...` | the loss function $J$ |
| `loss.backward()` | backpropagation: the chain rule computes $\partial J / \partial \mathbf{W}$ for every parameter |
| `opt.step()` | one gradient-descent update, $\mathbf{W} \leftarrow \mathbf{W} - \eta\, \partial J / \partial \mathbf{W}$ (Adam does a smarter version, Lecture 6) |

The function also records the training and validation loss during training (in `model.history`), so
that we can plot the loss curves. `extra_loss` is an optional function of `h` that is added to the
loss; we use it in Part 2.

In [ ]:
def train_ae(model, X=X_train, epochs=30, lr=1e-3, weight_decay=0.0, extra_loss=None, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.history = {'epoch': [], 'train': [], 'val': []}
    for epoch in range(1, epochs + 1):
        model.train()
        for b in batches(len(X), 64):
            xb = X[b]
            ???                                   # 1. reset the gradients
            x_hat, h = ???                        # 2. forward pass
            loss = ???                            # 3. reconstruction loss (compare with the INPUT)
            if extra_loss is not None:
                loss = loss + extra_loss(h)
            ???                                   # 4. backpropagation
            ???                                   # 5. gradient-descent step
        if epoch % max(1, epochs // 50) == 0 or epoch == 1:      # record the loss curves (about 50 points)
            model.eval()
            with torch.no_grad():
                model.history['epoch'].append(epoch)
                model.history['train'].append(recon_loss(model(X)[0], X).item())
                model.history['val'].append(recon_loss(model(X_test)[0], X_test).item())
            if verbose and epoch % (max(1, epochs // 3) // max(1, epochs // 50) * max(1, epochs // 50)) == 0:
                print(f'epoch {epoch:5d}   train loss {model.history["train"][-1]:7.2f}   validation loss {model.history["val"][-1]:7.2f}')
    model.eval()
    return model

In [ ]:
# Test for Task 1b: two epochs should already reduce the validation loss clearly
set_seed(0)
_m = ShallowAE(196).to(device)
with torch.no_grad():
    _before = recon_loss(_m(X_test)[0], X_test).item()
train_ae(_m, epochs=2, verbose=False)
print(f'validation loss before: {_before:.1f}   after 2 epochs: {_m.history["val"][-1]:.1f}')
assert _m.history['val'][-1] < 0.5 * _before, 'the loss did not decrease: check the five lines'
print('✅ Task 1b passed')

### Experiment 1: can an autoencoder overfit?

An autoencoder has no labels, but it can still **overfit** (Lecture 2): it can learn to reconstruct
its *training* images very well without learning anything that carries over to new images. To make
this happen, we give it very little data: only **100 training images**, and many epochs (1,000
epochs, about 2,000 gradient steps), with no regularisation. We use $H = 196$ hidden units
($196 = 14 \times 14$, so the weights fit in a square grid). Note that there are now more hidden units
than training images.

The validation loss is computed on the 2,000 validation images (`X_test`), which the model never sees
during training.

✍️ **Prediction:** sketch the training and validation loss curves you expect.

In [ ]:
def plot_curves(models, ax=None, title=None):
    # Training (solid) and validation (dashed) loss curves for one or more models.
    ax = ax or plt.figure(figsize=(6, 4)).gca()
    for i, (name, m) in enumerate(models.items()):
        h = m.history
        ax.plot(h['epoch'], h['train'], c=f'C{i}', label=f'{name}: train')
        ax.plot(h['epoch'], h['val'], c=f'C{i}', ls='--', label=f'{name}: validation')
    ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('reconstruction loss (log scale)')
    ax.legend(fontsize=8)
    if title:
        ax.set_title(title)


X_small = X_train[:100]
set_seed(0)
t0 = time.time()
ae_small = train_ae(ShallowAE(196).to(device), X=X_small, epochs=1000)
print(f'Training time: {time.time() - t0:.0f} s')
plot_curves({'100 images': ae_small}, title='Shallow AE trained on 100 images')
plt.show()

with torch.no_grad():
    show_images(torch.cat([X_small[:10], ae_small(X_small[:10])[0], X_test[:10], ae_small(X_test[:10])[0]]), nrow=10,
                title='training images and their reconstructions  ·  validation images and their reconstructions', figsize=(8, 3.6))

### 🛠️ TASK 1c: What have the hidden units learned?

Hidden unit $j$ computes $h_j = \sigma(\mathbf{w}_j^\top \mathbf{x} + b_j)$, where $\mathbf{w}_j$ is
**row $j$** of $\mathbf{W}_1$. Recall from Lecture 2 that the inner product $\mathbf{w}_j^\top\mathbf{x}$
measures how **similar** $\mathbf{x}$ is to $\mathbf{w}_j$. So if we reshape $\mathbf{w}_j$ (784
values) into a 28×28 image, we see the pattern that unit $j$ responds to most strongly: of all
inputs with a fixed norm, $h_j$ is largest when $\mathbf{x}$ points in the direction of $\mathbf{w}_j$.
(You did the same with softmax regression in Lab 3.)

Fill in the two lines: get the weight matrix of the **first** layer of the encoder, and reshape it
so that every row becomes a 1×28×28 image.

✍️ **Prediction:** what do you expect the weight images of the overfitted model to look like?

In [ ]:
def show_weights(model, title, ax=None, n=100):
    W = ???                                           # the (n_hidden, 784) weight matrix of the encoder
    W = W.detach().cpu()
    W = (W - W.mean(1, keepdim=True)) / W.std(1, keepdim=True)   # normalise each unit's image for display
    images = ???                                      # reshape to (n_hidden, 1, 28, 28)
    grid = torchvision.utils.make_grid(images[:n], nrow=10, normalize=True, scale_each=True, padding=1)
    ax = ax or plt.figure(figsize=(6, 6)).gca()
    ax.imshow(grid[0], cmap='gray'); ax.axis('off'); ax.set_title(title)


show_weights(ae_small, '100 images: weights of the first 100 hidden units')

### Questions 1.1

1. How do the loss curves show that the model overfits? Compare the reconstructions of the training
   images with those of the validation images.
2. Describe the weight images. Do they look like digits, strokes, or something else?
3. Why does overfitting show up as **noise** in the weights? Two hints:
   * with 196 hidden units and only 100 images, how many parameters are there per training image?
   * by the chain rule (Lecture 3), the gradient of the loss with respect to the weight $w_{ji}$
     between pixel $i$ and hidden unit $j$ is proportional to the pixel value $x_i$. What happens to
     the weights of pixels that are 0 (black) in **all** 100 training images, e.g. near the border?

### Two remedies: more data, and weight decay

We now change **one thing at a time** compared with the overfitted model:

* **More data:** all 6,000 training images (30 epochs, about 2,800 gradient steps: a similar amount of
  training).
* **Weight decay** (Lecture 2): an L2 penalty on the weights. In the optimiser it subtracts a small
  multiple of each weight at every step, so weights that are not supported by the data **shrink
  towards zero**.

Together with the overfitted model, that gives a 2×2 comparison.

✍️ **Prediction:** which remedy will close the gap between the training and validation loss? Which
one will make the weight images look less noisy?

In [ ]:
runs = {'100 images': ae_small}
for name, X, epochs, wd in [('100 images + weight decay', X_small, 1000, 3e-3),
                            ('6,000 images', X_train, 30, 0.0),
                            ('6,000 images + weight decay', X_train, 30, 3e-3)]:
    set_seed(0)
    print(f'--- {name}')
    runs[name] = train_ae(ShallowAE(196).to(device), X=X, epochs=epochs, weight_decay=wd)

print(f'\n{"":30s}{"train loss":>12s}{"validation loss":>17s}')
for name, m in runs.items():
    print(f'{name:30s}{m.history["train"][-1]:12.1f}{m.history["val"][-1]:17.1f}')

fig, axs = plt.subplots(2, 4, figsize=(22, 10), gridspec_kw={'height_ratios': [1.3, 1]})
for j, (name, m) in enumerate(runs.items()):
    show_weights(m, name, ax=axs[0, j])
    plot_curves({name: m}, ax=axs[1, j])
plt.tight_layout(); plt.show()

### Questions 1.2

1. Which remedy closes the gap between the training and validation loss? Which remedy removes the
   noise from the weight images? Are they the same?
2. Explain why weight decay cleans up the weights, using your answer to Question 1.1.3.
3. With 6,000 images, weight decay makes the validation loss slightly **worse**. Why can a regulariser
   hurt when the model is not overfitting?
4. Lab 3 said: "The best kind of regularization is adding more data." Do your results agree? What would
   you do in your project if you cannot get more data?

### The reference model for the rest of the notebook

From now on, we use the model trained on **6,000 images with weight decay** as our reference
(`ae_dense`): it doesn't overfit, and its clean weight images make the effect of the sparsity penalty
in Part 2 easiest to see. Let's look at its **activations**: for a given test image, how many of the
196 hidden units are "on"?

In [ ]:
ae_dense = runs['6,000 images + weight decay']


@torch.no_grad()
def activation_stats(model):
    h = model(X_test)[1]
    return {'recon. loss': round(recon_loss(model(X_test)[0], X_test).item(), 2),
            'mean activation': round(h.mean().item(), 3),
            'fraction of units > 0.5': round((h > 0.5).float().mean().item(), 3)}


print(activation_stats(ae_dense))
with torch.no_grad():
    h = ae_dense(X_test)[1].cpu()
plt.figure(figsize=(10, 3))
plt.subplot(121); plt.hist(h.flatten(), bins=50); plt.xlabel('activation $h_j$'); plt.title('All activations (test set)')
plt.subplot(122); plt.bar(range(196), h[0]); plt.xlabel('hidden unit j'); plt.title('Activations for ONE test image')
plt.tight_layout(); plt.show()

### Questions 1.3

1. Look at the weight images of the reference model. Can you tell what an individual hidden unit
   "detects"?
2. Look at the activations. Roughly how many hidden units take part in representing a single image?
   Is this a *local* code (one unit per image type) or a *distributed* code (each image is a
   combination of many units)?
3. Is this autoencoder undercomplete or overcomplete? What could go wrong with $H = 1000$ hidden
   units and no regularisation?
4. The reconstructions are good and the model doesn't overfit. So why are we not happy with the
   learned features?

---
# Part 2 — A sparse autoencoder

A **sparse autoencoder** uses the same architecture, but adds a penalty that keeps most hidden units
close to zero for any given input (lecture, Section 2.5). Here is how it works in practice.

**Step 1: measure how active each unit is.** For hidden unit $j$, compute its average activation over
the $n$ images in a batch:

$$\hat\rho_j = \frac{1}{n}\sum_i h_j(\mathbf{x}^{(i)}).$$

**Step 2: penalise deviations from a small target** $\rho$ (here $\rho = 0.1$: each unit should be
"on" for about 10% of the images). Because sigmoid activations lie between 0 and 1, we can think of
$\hat\rho_j$ and $\rho$ as the probabilities of a coin landing heads, and measure how different the
two coins are with the **KL divergence** (you will meet KL divergence again in the VAE in 9C and in
Lecture 13):

$$\mathrm{KL}(\rho \,\|\, \hat\rho_j) = \rho \log\frac{\rho}{\hat\rho_j} + (1-\rho)\log\frac{1-\rho}{1-\hat\rho_j}.$$

It is zero when $\hat\rho_j = \rho$, and grows quickly as $\hat\rho_j$ moves away from $\rho$ (try a
few values on paper).

**Step 3: add it to the loss**, summed over all hidden units and weighted by a hyperparameter $\lambda$:

$$J_\text{sparse} = J + \lambda \sum_j \mathrm{KL}(\rho \,\|\, \hat\rho_j).$$

> **Note:** the code in the lecture notes uses a simpler alternative, an **L1 penalty** on the
> activations (`h.abs().mean()`). It pushes in the same direction (activations towards zero), but
> doesn't control the target activity $\rho$ directly.

### 🛠️ TASK 2a: Implement the sparsity penalty

In [ ]:
def sparsity_penalty(h, rho=0.1):
    rho_hat = ???                                     # average activation of each hidden unit over the batch: shape (n_hidden,)
    rho_hat = rho_hat.clamp(1e-6, 1 - 1e-6)           # avoid log(0)
    kl = ???                                          # KL(rho || rho_hat) for every unit: shape (n_hidden,)
    return kl.sum()

In [ ]:
# Test for Task 2a
_h = torch.full((32, 196), 0.1)
assert abs(sparsity_penalty(_h).item()) < 1e-5, 'the penalty should be 0 when every unit is active exactly 10% of the time'
_h = torch.full((32, 196), 0.5)
assert abs(sparsity_penalty(_h).item() - 196 * (0.1 * math.log(0.2) + 0.9 * math.log(1.8))) < 1e-2, 'wrong value for rho_hat = 0.5'
_h = torch.rand(32, 196)
assert abs(sparsity_penalty(_h).item() - sparsity_penalty(_h[torch.randperm(32)]).item()) < 1e-4, 'average over the batch (dim=0), not over the units'
print('✅ Task 2a passed')

### Turn the sparsity dial

We train the same network twice more, changing **only** $\lambda$: a moderate penalty
($\lambda = 0.3$) and a strong one ($\lambda = 3$). Everything else (architecture, $\rho$, weight
decay, epochs, seed) is unchanged, so any difference is caused by the penalty.

✍️ **Prediction** (write it down before running the cell):
* **Mechanism:** what happens to the mean activation and to the fraction of active units?
* **Outcome:** what happens to the reconstruction loss?
* What will the weight images look like for $\lambda = 0.3$ and for $\lambda = 3$?

In [ ]:
models = {'λ = 0 (reference)': ae_dense}
for lam in [0.3, 3.0]:
    set_seed(0)
    print(f'--- λ = {lam}')
    models[f'λ = {lam}'] = train_ae(ShallowAE(196).to(device), weight_decay=3e-3,
                                   extra_loss=lambda h, lam=lam: lam * sparsity_penalty(h))

print()
for name, m in models.items():
    print(f'{name:16s}', activation_stats(m))

fig, axs = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, (name, m) in zip(axs, models.items()):
    show_weights(m, name, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# How does each model represent ONE test image? Compare the activation patterns.
fig, axs = plt.subplots(1, 3, figsize=(15, 2.5), sharey=True)
for ax, (name, m) in zip(axs, models.items()):
    with torch.no_grad():
        ax.bar(range(196), m(X_test[:1])[1][0].cpu())
    ax.set_title(name); ax.set_xlabel('hidden unit j')
plt.tight_layout(); plt.show()

### Questions 2

1. **Mechanism:** did the penalty do what it was designed to do? Use the numbers.
2. **Outcome:** what did it cost in reconstruction quality? Why is that the expected price?
3. Describe the weight images for $\lambda = 0.3$ and for $\lambda = 3$. What does a typical unit
   detect in each case?
4. With $\lambda = 3$, only a handful of units are active per image. Explain why the units then end up
   looking like **whole digits**. Which classical method from Lecture 2 does the extreme case (exactly
   one active unit per image) remind you of?
5. Why do sparse features tend to be more interpretable, and why might they generalise better?
6. In your project, would you rather have the features of $\lambda = 0$, $0.3$ or $3$? (There is no
   single right answer: argue from what you would use the features for.)

---
# Part 3 — A 2D embedding with a CNN classifier

Autoencoders learn a representation **without labels**. Before we look at the latent space of a
convolutional autoencoder in 9B, it is useful to see what a representation learned **with labels**
looks like. We train a CNN classifier, but force all information to pass through a layer with only
**two** neurons. These two numbers are an **embedding** of the image that we can plot directly.

### 🔗 Theory ↔ code: tracking shapes

For a convolution with filter size $F$, stride $S$ and no padding, the output size is
$(N - F)/S + 1$ (Lecture 4). A 2×2 max pool with stride 2 halves the size (rounding down).

✍️ **Prediction:** write down the shape of the feature map after each layer of `features` below, for a
1×28×28 input. What is the shape just before the embedding layer?

In [ ]:
class EmbeddingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU())
        self.embed = nn.Sequential(???, nn.Flatten())     # 32 x 3 x 3  ->  2 x 1 x 1 with a tanh, then flatten to (N, 2)
        self.classifier = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, 10))                             # logits: no softmax (CrossEntropyLoss applies it)

    def forward(self, x):
        z = self.embed(self.features(x))
        return self.classifier(z), z

In [ ]:
# Test for Task 3: print the shapes layer by layer, and check the embedding
_m = EmbeddingCNN().to(device)
_x = X_train[:4]
for layer in list(_m.features) + list(_m.embed):
    _x = layer(_x)
    print(f'{layer.__class__.__name__:10s} -> {tuple(_x.shape)}')
_logits, _z = _m(X_train[:4])
assert _z.shape == (4, 2), f'the embedding should have shape (N, 2), not {tuple(_z.shape)}'
assert _z.abs().max() <= 1, 'did you add the tanh?'
assert _logits.shape == (4, 10)
print('✅ Task 3 passed')

### Train the classifier

In [ ]:
set_seed(0)
cnn = EmbeddingCNN().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=3e-3)
t0 = time.time()
for epoch in range(30):
    cnn.train()
    for b in batches(len(X_train), 64):
        logits, _ = cnn(X_train[b])
        loss = F.cross_entropy(logits, y_train[b])      # this time we DO use the labels
        opt.zero_grad(); loss.backward(); opt.step()
    if (epoch + 1) % 10 == 0:
        cnn.eval()
        with torch.no_grad():
            acc = (cnn(X_test)[0].argmax(1) == y_test).float().mean().item()
        print(f'epoch {epoch + 1:3d}   test accuracy {acc:.1%}')
print(f'Training time: {time.time() - t0:.0f} s')

In [ ]:
def plot_embedding(z, labels, ax=None, title=None):
    z, labels = z.detach().cpu().numpy(), labels.cpu().numpy()
    ax = ax or plt.gca()
    for c in range(10):
        ax.scatter(z[labels == c, 0], z[labels == c, 1], s=4, label=str(c), color=plt.cm.tab10(c))
    ax.legend(markerscale=4, loc='center left', bbox_to_anchor=(1, 0.5))
    ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]')
    if title:
        ax.set_title(title)


cnn.eval()
with torch.no_grad():
    z_cnn = cnn(X_test)[1]
plt.figure(figsize=(7, 6))
plot_embedding(z_cnn, y_test, title='CNN classifier: 2D embedding of 2,000 test images')
plt.show()

### Questions 3

1. We have compressed 784 pixels to **two** numbers, and the classes can still be told apart. Which
   classes are well separated, and which overlap? Do the overlaps make sense?
2. Why does a $\tanh$ on the embedding make the plot work better? Think about what the cross-entropy
   loss "wants" to do with the embedding vectors if they are not bounded.
3. This embedding was learned **with labels**. Predict how the 2D latent space of an autoencoder
   (trained without labels, in 9B) will differ.

---
# Part 4 — Exam-style questions

The written exam uses multiple-choice questions like these. Answer them without running any code.

**Q1.** An autoencoder for 28×28 images has a single hidden layer with 1,000 units and no
regularisation. Which statement is correct?

* (a) It is undercomplete, so it is forced to learn the most important features.
* (b) It is overcomplete, so it may learn (close to) the identity mapping without learning useful structure.
* (c) It cannot be trained without labels.
* (d) It is a sparse autoencoder, because most of its units will be zero.

**Q2.** In a shallow autoencoder, `encoder[0].weight` has shape (196, 784). What does the image
obtained by reshaping row $j$ to 28×28 show?

* (a) The reconstruction of training image $j$.
* (b) The input pattern that hidden unit $j$ responds to most strongly.
* (c) The gradient of the loss with respect to image $j$.
* (d) The average of all training images.

**Q3.** The sparsity penalty $\sum_j \mathrm{KL}(\rho \,\|\, \hat\rho_j)$ with $\rho = 0.05$ encourages…

* (a) every hidden unit to be active (close to 1) for about 5% of the inputs.
* (b) 5% of the weights to be exactly zero.
* (c) the reconstruction error to be below 5%.
* (d) the latent vectors to follow a standard normal distribution.

**Q4.** A 3×3 convolution with 2 filters, stride 1 and no padding is applied to a feature map of shape
32×3×3. What is the output shape?

* (a) 2×3×3  · (b) 32×1×1  · (c) 2×1×1  · (d) 2×5×5

---
**Next: notebook 9B**, where we replace the fully connected layers with convolutions, look at the latent
space of an autoencoder, and use it for denoising.